In [1]:
# ============================================================
# 1. IMPORTS
# ============================================================
import pandas as pd
import numpy as np

# ============================================================
# 2. CARREGAMENTO DO DATASET
# ============================================================
# Ajuste o caminho conforme seu arquivo
df = pd.read_csv(f'dataset/online_retail.csv')

# Visualizar algumas linhas
df.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# ============================================================
# 3. TRATAMENTO INICIAL (SE NECESSÁRIO)
# ============================================================

#Padronizando nomes de colunas
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

# Garantir que quantity e unitprice são numéricos
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unitprice'] = pd.to_numeric(df['unitprice'], errors='coerce')

# Remover linhas com quantity <= 0 ou unitprice <= 0
df = df[(df['quantity'] > 0) & (df['unitprice'] > 0)]

# Remover NaN em stockcode ou invoiceno
df = df.dropna(subset=['stockcode', 'invoiceno'])

df.head()


,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
# ============================================================
# 4. CRIAÇÃO DA RECEITA DA LINHA
# ============================================================

df['receita'] = df['quantity'] * df['unitprice']
df.head()


,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,receita
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [4]:
# ============================================================
# 5. AGRUPAR POR PRODUTO E SOMAR A RECEITA TOTAL
# ============================================================

df_prod = (
    df.groupby('stockcode', as_index=False)
      .agg(receita_total=('receita', 'sum'))
      .sort_values('receita_total', ascending=False)
)

df_prod.head()


,stockcode,receita_total
3911,DOT,206248.77
1310,22423,174484.74
2465,23843,168469.60
3407,85123A,104518.80
2670,47566,99504.33


In [5]:
# ============================================================
# 6. CALCULAR PERCENTUAL, CUMULATIVO E CATEGORIA ABC
# ============================================================

df_prod['contrib_pct'] = df_prod['receita_total'] / df_prod['receita_total'].sum()
df_prod['cum_pct'] = df_prod['contrib_pct'].cumsum()

df_prod['categoria_ABC'] = pd.cut(
    df_prod['cum_pct'],
    bins=[0, 0.80, 0.95, 1.0],
    labels=['A', 'B', 'C']
)

df_prod.head()


,stockcode,receita_total,contrib_pct,cum_pct,categoria_ABC
3911,DOT,206248.77,0.019336,0.019336,A
1310,22423,174484.74,0.016358,0.035694,A
2465,23843,168469.60,0.015794,0.051488,A
3407,85123A,104518.80,0.009799,0.061286,A
2670,47566,99504.33,0.009329,0.070615,A


In [6]:
# ============================================================
# 7. MÉTRICAS PRINCIPAIS DA CURVA ABC
# ============================================================

receita_total_geral = df_prod['receita_total'].sum()

categoria_a_pct = df_prod[df_prod['categoria_ABC'] == 'A']['contrib_pct'].sum() * 100
qtde_prod_a = df_prod[df_prod['categoria_ABC'] == 'A'].shape[0]

categoria_c_pct = df_prod[df_prod['categoria_ABC'] == 'C']['contrib_pct'].sum() * 100

print(f"Contribuição da Categoria A: {categoria_a_pct:.2f}%")
print(f"Quantidade de Produtos A: {qtde_prod_a}")
print(f"Contribuição da Categoria C: {categoria_c_pct:.2f}%")


Contribuição da Categoria A: 79.98%
Quantidade de Produtos A: 803
Contribuição da Categoria C: 5.01%


In [7]:
# ============================================================
# 8. MERGE DO ABC COM O DATASET ORIGINAL (PARA USAR NO BI)
# ============================================================

df_final = df.merge(
    df_prod[['stockcode','contrib_pct','cum_pct','categoria_ABC']],
    on='stockcode',
    how='left'
)

df_final.head()


,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,receita,contrib_pct,cum_pct,categoria_ABC
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,0.009799,0.061286,A
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,0.000791,0.559938,A
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,0.000718,0.581592,A
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,0.001515,0.388249,A
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,0.002076,0.301201,A


In [8]:
# ============================================================
# 9. VERIFICAR SE A BASE FINAL ESTÁ OK
# ============================================================

print("Duplicados por InvoiceNo + StockCode:")
print(df_final[['invoiceno','stockcode']].duplicated().sum())

df_final.info()


Duplicados por InvoiceNo + StockCode:
10502
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 530104 entries, 0 to 530103
Data columns (total 12 columns):
 #   Column         Non-Null Count   Dtype   
---  ------         --------------   -----   
 0   invoiceno      530104 non-null  object  
 1   stockcode      530104 non-null  object  
 2   description    530104 non-null  object  
 3   quantity       530104 non-null  int64   
 4   invoicedate    530104 non-null  object  
 5   unitprice      530104 non-null  float64 
 6   customerid     397884 non-null  float64 
 7   country        530104 non-null  object  
 8   receita        530104 non-null  float64 
 9   contrib_pct    530104 non-null  float64 
 10  cum_pct        530104 non-null  float64 
 11  categoria_ABC  530101 non-null  category
dtypes: category(1), float64(5), int64(1), object(5)
memory usage: 45.0+ MB


In [9]:
df_final.to_csv("datasetCurva_ABC.csv", index=False)